# Experiment: First Quote Acceptance Gate

Objective:
- Convert the June 10 quote-acceptance rule into a tiny executable gate.
- Confirm HbF-only offers fail after the June 9 HBG benchmark refresh.
- Keep all examples public-safe and preclinical.

In [ ]:
from __future__ import annotations

PASS = "quote_acceptance_ready"
HOLD = "quote_acceptance_partial_endpoint_hold"
REJECT = "quote_acceptance_rejected"

REQUIRED_FIELDS = [
    "recipient_class_passed",
    "model_relevance",
    "controls",
    "hbf_or_hbg",
    "maturation",
    "viability",
    "alpha_or_autophagy",
    "hemolysis_or_membrane_safety",
    "raw_data",
    "cost",
    "timeline",
    "ethics",
    "biosafety",
    "material_constraints",
]

BIOLOGY_ENDPOINTS_BEYOND_HBF = [
    "maturation",
    "viability",
    "alpha_or_autophagy",
    "hemolysis_or_membrane_safety",
]

BLOCKED_ROUTES = [
    "private_records",
    "patient_samples",
    "treatment_selection",
    "trial_screening",
    "purchase",
    "import",
    "procurement",
]

## Gate Logic

Reject blocked clinical or private-data routes first. Then reject HbF-only offers,
hold incomplete multi-endpoint offers, and pass only complete public-safe offers.

In [ ]:
def classify_quote_offer(offer: dict[str, bool]) -> dict[str, object]:
    """Classify a public-safe preclinical quote offer."""
    blocked = [field for field in BLOCKED_ROUTES if offer.get(field)]
    if blocked:
        return {"decision": REJECT, "reason": "blocked_route", "fields": blocked}

    if offer.get("recipient_class_passed") is False:
        return {"decision": REJECT, "reason": "recipient_not_qualified", "fields": []}

    if not offer.get("hbf_or_hbg"):
        return {
            "decision": REJECT,
            "reason": "missing_hbf_or_hbg",
            "fields": ["hbf_or_hbg"],
        }

    has_biology_beyond_hbf = any(
        offer.get(field) for field in BIOLOGY_ENDPOINTS_BEYOND_HBF
    )
    if not has_biology_beyond_hbf:
        return {
            "decision": REJECT,
            "reason": "hbf_only_offer",
            "fields": BIOLOGY_ENDPOINTS_BEYOND_HBF,
        }

    missing = [field for field in REQUIRED_FIELDS if not offer.get(field)]
    if missing:
        return {
            "decision": HOLD,
            "reason": "missing_required_fields",
            "fields": missing,
        }

    return {"decision": PASS, "reason": "complete_public_safe_offer", "fields": []}

## Smoke Cases

These are synthetic offer labels, not real lab replies and not treatment advice.

In [ ]:
base_offer = {field: True for field in REQUIRED_FIELDS}
base_offer.update({field: False for field in BLOCKED_ROUTES})

offers = {
    "hbf_only": base_offer
    | {
        "maturation": False,
        "viability": False,
        "alpha_or_autophagy": False,
        "hemolysis_or_membrane_safety": False,
    },
    "partial_endpoint": base_offer
    | {
        "alpha_or_autophagy": False,
        "hemolysis_or_membrane_safety": False,
    },
    "private_record_request": base_offer | {"private_records": True},
    "complete_public_safe": base_offer,
}

results = {name: classify_quote_offer(offer) for name, offer in offers.items()}
results

In [ ]:
assert results["hbf_only"]["decision"] == REJECT
assert results["partial_endpoint"]["decision"] == HOLD
assert results["private_record_request"]["decision"] == REJECT
assert results["complete_public_safe"]["decision"] == PASS

decision_summary = {name: result["decision"] for name, result in results.items()}
decision_summary

## Decision

- Continue using the June 10 acceptance gate.
- Reject HbF-only offers after the June 9 benchmark refresh.
- Hold partial offers until missing endpoints, raw data, cost, timeline, and
  safety boundaries are explicit.